# Week 5 — EDA I: distributions, missingness, outliers

### Deliverable
- 6–8 plots + 5 insights (each with a 'could be wrong because...')


In [ ]:
# Core imports (kept minimal for beginners)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

# Dataset URL (UCI Heart Disease - Cleveland)
UCI_URL = "https://www.archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

# Column names for processed.cleveland.data (14 columns commonly used in teaching)
COLS = ["age","sex","cp","trestbps","chol","fbs","restecg","thalach",
        "exang","oldpeak","slope","ca","thal","num"]


In [ ]:
import ssl
import io
import urllib.request # Added this import

def load_raw():
    # Create an unverified SSL context to bypass certificate verification
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE

    # Open the URL with the unverified context
    with urllib.request.urlopen(UCI_URL, context=ctx) as url_response:
        # Read the content and decode it
        s = url_response.read().decode('utf-8')

    # Use io.StringIO to make the string behave like a file for pandas.read_csv
    df_raw = pd.read_csv(io.StringIO(s), header=None, names=COLS)
    return df_raw

def coerce_types(df_raw):
    # Missing values are sometimes encoded as "?"
    df = df_raw.replace("?", np.nan).copy()

    # Convert numeric-looking columns
    numeric_cols = ["age","trestbps","chol","thalach","oldpeak","ca","thal","num"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Binary target: disease present if num > 0 (UCI convention)
    df["disease"] = (df["num"] > 0).astype("Int64")
    return df

df_raw = load_raw()
df = coerce_types(df_raw)

df.head()


In [ ]:
def clean_heart(df_raw):
    df = df_raw.replace("?", np.nan).copy()
    numeric_cols = ["age","trestbps","chol","thalach","oldpeak","ca","thal","num"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["disease"] = (df["num"] > 0).astype("Int64")
    return df

df_clean = clean_heart(df_raw)


## Missingness profile


In [ ]:
miss = df_clean.isna().mean().sort_values(ascending=False)
display(miss.to_frame("missing_fraction"))

plt.figure()
miss.head(12).sort_values().plot(kind="barh")
plt.xlabel("missing fraction")
plt.title("Top missing columns")
plt.show()


## Numeric distributions (choose 4–5)


In [ ]:
numeric = ["age","trestbps","chol","thalach","oldpeak"]
for c in numeric:
    plt.figure()
    df_clean[c].dropna().plot(kind="hist", bins=20)
    plt.title(f"Distribution: {c}")
    plt.xlabel(c)
    plt.ylabel("count")
    plt.show()


## Outlier investigation (IQR rule as a starting point)


In [ ]:
col = "chol"  # TODO
x = df_clean[col].dropna()
q1, q3 = x.quantile(0.25), x.quantile(0.75)
iqr = q3 - q1
lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr

print("IQR bounds:", lo, hi)
outliers = df_clean[df_clean[col].between(lo, hi) == False][[col,"age","sex","disease"]].head(15)
outliers


## TODO — Write 5 insights
Insight 1

Observation: Age distribution is roughly normal, with the highest concentration around 55–60 years.
Could be wrong because:

· This is hospital data; younger or older populations might be underrepresented.
· The demographic structure at the time of data collection may differ from today.

Insight 2

Observation: thalach (maximum heart rate) distribution is left‑skewed; lower values appear to be associated with higher disease rates (though not shown directly here).
Could be wrong because:

· Distribution shape alone does not imply causation; low heart rate could be due to medication.
· Missing values in other variables might bias the perceived relationship.

Insight 3

Observation: oldpeak (ST depression during exercise) is heavily concentrated near 0, with very few high values.
Could be wrong because:

· Measurement precision or recording errors might affect extreme values.
· Different clinics may use different criteria for ST depression.

Insight 4

Observation: ca (number of major vessels) has ~8% missing values; these missing values may not be random (patients who did not undergo fluoroscopy).
Could be wrong because:

· Assuming missing completely at random (MCAR) would be misleading.
· Possibly patients with ca = 0 were less likely to be tested, causing informative missingness.

Insight 5

Observation: Cholesterol (chol) has a few extreme outliers above 350 mg/dL. Removing them reduces the difference between mean and median.
Could be wrong because:

· Outliers may represent real clinical cases (familial hypercholesterolemia) – discarding them would be incorrect.
· The IQR rule may be too strict or too loose for this dataset; a z‑score or domain‑based method might be more appropriate.
